In [17]:
#pip install medmnist

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim

import medmnist
from medmnist import PneumoniaMNIST

import torchvision.transforms as transforms
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score

In [19]:
# Transformation for converting to PyTorch tensors
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])


In [20]:
# 1. Download and load the datasets
train_dataset = PneumoniaMNIST(split='train', transform=data_transform, download=True)
val_dataset = PneumoniaMNIST(split='val', transform=data_transform, download=True)
test_dataset = PneumoniaMNIST(split='test', transform=data_transform, download=True)



In [21]:
# 2. Wrap in PyTorch DataLoaders for CNN model
batch_size = 64
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)


In [22]:
# Checking datasets
print(("Training Dataset:"), len(train_dataset))
print(("Validation Dataset:"),len(val_dataset))
print(("Test Dataset:"),len(test_dataset))

Training Dataset: 4708
Validation Dataset: 524
Test Dataset: 624


In [23]:
# CNN is the deep learning model of choice and 2D will be used instead of 1D since we are dealing with images and not sequences
class PneumoniaCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(1,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(128*3*3,128),
            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(128,1)
        )

    def forward(self,x):

        x=self.features(x)
        x=self.classifier(x)

        return x

In [24]:
# initializing the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PneumoniaCNN().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [25]:
# training loop
epochs = 10

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images,labels in train_loader:

        images = images.to(device)

        labels = labels.float().to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}")

    print("Training Loss:",
          running_loss/len(train_loader))

Epoch 1/10
Training Loss: 0.37907603181697225
Epoch 2/10
Training Loss: 0.18165554216987378
Epoch 3/10
Training Loss: 0.13644299711528662
Epoch 4/10
Training Loss: 0.12598179018980749
Epoch 5/10
Training Loss: 0.10966705243933846
Epoch 6/10
Training Loss: 0.09774131549371255
Epoch 7/10
Training Loss: 0.08652661559549538
Epoch 8/10
Training Loss: 0.0846572263871093
Epoch 9/10
Training Loss: 0.0658722152379719
Epoch 10/10
Training Loss: 0.06312262575217598


In [26]:
# validation
model.eval()

predictions=[]
truth=[]

with torch.no_grad():

    for images,labels in val_loader:

        images=images.to(device)

        outputs=model(images)

        preds=(torch.sigmoid(outputs)>0.5).cpu().numpy()

        predictions.extend(preds.flatten())

        truth.extend(labels.numpy().flatten())

val_accuracy = accuracy_score(
    truth,
    predictions
)

print("Validation Accuracy:",val_accuracy)

Validation Accuracy: 0.9694656488549618


In [27]:
# test set evaluation
model.eval()

predictions=[]
truth=[]

with torch.no_grad():

    for images,labels in test_loader:

        images=images.to(device)

        outputs=model(images)

        preds=(torch.sigmoid(outputs)>0.5).cpu().numpy()

        predictions.extend(preds.flatten())

        truth.extend(labels.numpy().flatten())

test_accuracy = accuracy_score(
    truth,
    predictions
)

print("Test Accuracy:",test_accuracy)

Test Accuracy: 0.8413461538461539


In [28]:
# report for classification
from sklearn.metrics import classification_report

print(classification_report(
    truth,
    predictions,
    target_names=["Normal","Pneumonia"]
))

              precision    recall  f1-score   support

      Normal       0.97      0.59      0.74       234
   Pneumonia       0.80      0.99      0.89       390

    accuracy                           0.87       624
   macro avg       0.90      0.83      0.85       624
weighted avg       0.89      0.87      0.87       624

